# UAV-Sentinel-2 S2A

## UAV data to Sentinel-2 S2A data

In [ ]:
#数据转换
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

df = pd.read_csv(r"\data\input\UAV.csv", encoding='gbk')
uav = df.iloc[:, :164]  # First 164 columns = spectral bands (UAV hyperspectral)
# Remaining columns = auxiliary variables (e.g., water quality, metadata)
other_cols = df.iloc[:, 164:]
srf = pd.read_csv(r"\SRF\Sentinel-2 S2A.csv", encoding='gbk')

# UAV wavelength array
uav_wl = uav.columns.astype(float).values

# UAV spectral matrix
uav_spec = uav.values

# Sentinel-2 SRF wavelength
srf_wl = srf["SR_WL"].values

def uav_to_s2(spectrum,
              uav_wavelengths,
              srf_df):
    """
    Simulate Sentinel-2 band reflectance from UAV hyperspectral data.

    Parameters
    ----------
    spectrum : np.ndarray
        UAV spectrum (1D array)
    uav_wavelengths : np.ndarray
        Wavelength vector of UAV sensor
    srf_df : pd.DataFrame
        Sentinel-2 spectral response function table

    Returns
    -------
    dict
        Simulated Sentinel-2 band reflectance
    """

    # Interpolate UAV spectrum to SRF wavelength grid
    interp_func = interp1d(
        uav_wavelengths,
        spectrum,
        kind='linear',
        bounds_error=False,
        fill_value=0
    )

    spec_interp = interp_func(
        srf_df["SR_WL"].values
    )

    result = {}

    for band in srf_df.columns[1:]:

        response = srf_df[band].values

        if response.sum() == 0:
            continue

        value = np.trapz(
            spec_interp * response,
            srf_df["SR_WL"].values
        ) / np.trapz(
            response,
            srf_df["SR_WL"].values
        )

        result[band] = value

    return result

s2_results = []

for i in range(len(uav)):

    s2_results.append(
        uav_to_s2(
            uav_spec[i],
            uav_wl,
            srf
        )
    )

s2_df = pd.DataFrame(s2_results)

print(s2_df.head())

keep_bands = [
    'S2A_SR_AV_B1',
    'S2A_SR_AV_B2',
    'S2A_SR_AV_B3',
    'S2A_SR_AV_B4',
    'S2A_SR_AV_B5',
    'S2A_SR_AV_B6',
    'S2A_SR_AV_B7',
    'S2A_SR_AV_B8',
    'S2A_SR_AV_B8A',
    'S2A_SR_AV_B9'
]

s2_df = s2_df[keep_bands]
s2_df.columns = [
    'B1','B2','B3','B4',
    'B5','B6','B7',
    'B8','B8A','B9'
]
result = pd.concat(
    [s2_df, other_cols],
    axis=1
)
print(result.head())
print(result.shape)
result.to_csv(
    r"\data\input\UAV_Sentinel_S2A.csv",
    index=False,
    encoding='utf-8-sig'
)

C:\Users\PC\AppData\Local\Temp\ipykernel_44568\1998904868.py:55: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  value = np.trapz(
C:\Users\PC\AppData\Local\Temp\ipykernel_44568\1998904868.py:58: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  ) / np.trapz(


   S2A_SR_AV_B1  S2A_SR_AV_B2  S2A_SR_AV_B3  S2A_SR_AV_B4  S2A_SR_AV_B5  \
0      3.779268      6.314651     10.367480      9.072357     11.125766   
1      3.723377      6.333664     10.431520      8.944443     10.889991   
2      3.805880      6.337246     10.262945      8.846258     10.634643   
3      3.524588      6.134429     10.090588      8.454472     10.175934   
4      3.656097      6.259065     10.280785      8.449245     10.111942   

   S2A_SR_AV_B6  S2A_SR_AV_B7  S2A_SR_AV_B8  S2A_SR_AV_B8A  S2A_SR_AV_B9  \
0      4.586777      3.913552      4.183957       4.076732      1.503265   
1      4.395331      3.846816      3.978563       3.725088      0.376494   
2      4.388882      3.448933      3.667611       3.651779      0.269601   
3      4.053916      3.111808      3.169149       2.933654      0.201965   
4      3.987648      3.011982      2.974439       2.843491      0.027599   

   S2A_SR_AV_B10  S2A_SR_AV_B11  S2A_SR_AV_B12  
0            0.0            0.0            

In [5]:
s2_df

,S2A_SR_AV_B1,S2A_SR_AV_B2,S2A_SR_AV_B3,S2A_SR_AV_B4,S2A_SR_AV_B5,S2A_SR_AV_B6,S2A_SR_AV_B7,S2A_SR_AV_B8,S2A_SR_AV_B8A,S2A_SR_AV_B9,...,S2A_SR_AV_B11,S2A_SR_AV_B12,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,3.779268,6.314651,10.367480,9.072357,11.125766,4.586777,3.913552,4.183957,4.076732,1.503265,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.723377,6.333664,10.431520,8.944443,10.889991,4.395331,3.846816,3.978563,3.725088,0.376494,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.805880,6.337246,10.262945,8.846258,10.634643,4.388882,3.448933,3.667611,3.651779,0.269601,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3.524588,6.134429,10.090588,8.454472,10.175934,4.053916,3.111808,3.169149,2.933654,0.201965,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3.656097,6.259065,10.280785,8.449245,10.111942,3.987648,3.011982,2.974439,2.843491,0.027599,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
565,3.187951,4.944356,5.838774,2.675332,1.679466,0.171723,0.000540,0.014869,0.020818,0.000000,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
566,3.126226,4.981573,5.987303,2.664966,1.666872,0.069583,0.000027,0.007979,0.012819,0.000023,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
567,3.081787,4.909275,5.921132,2.686663,1.686333,0.162716,0.001066,0.008889,0.007166,0.000000,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
568,2.671599,4.292943,5.213139,2.381674,1.490117,0.097943,0.000083,0.005453,0.007576,0.000000,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## training

### CSF-FPE

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F

DATA_DIR = r"\data\input"
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")
model_save_dir = os.path.join(DATA_DIR, "UAV-Sentinel2S2A/CSF-FPE")
os.makedirs(model_save_dir, exist_ok=True)

#=========================
#⭐ Dataset selection
#=========================
selected_datasets = ["uav2s2"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
    "uav2s2": (3, "UAV_Sentinel_S2A.csv"),
    "uav2l8": (4, "UAV_Landsat8_SRF.csv")
}

# =========================
# 0. Wavelength normalization
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# ⭐ Unified split loader
# =========================
def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]
    return (
        np.array(split["train_idx"]),
        np.array(split["test_idx"])
    )

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1 or dtype == 3:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),
                     (740,15),(783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        elif dtype == 2:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=64, grid_size=64):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. Fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # Feature fusion layer
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 4. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # =========================
        # Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Linear(d_model//2, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # -------------------------
        # 2. FiLM conditioning
        # -------------------------
        cond = torch.stack([
            dtype.float(),
            temp,
            wl.mean(dim=1).squeeze(-1),
            delta.mean(dim=1).squeeze(-1)
        ], dim=1)

        gamma, beta = self.film(cond).chunk(2, dim=1)
        z = gamma.unsqueeze(1) * z + beta.unsqueeze(1)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        z = torch.cat([t, z], dim=1)          # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ evaluation
# =========================
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            y_true.extend(batch["y"].cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, mae, r2


# =========================
# ⭐ train with early stopping
# =========================
def train():

    device = "cuda" if torch.cuda.is_available() else "cpu"

    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # global normalization
    # =========================
    all_x = []
    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)
    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    dataset = SpectralDataset(paths[0][1], paths[0][0], x_mean, x_std)

    # =========================
    # ⭐ 5-fold split
    # =========================
    folds = json.load(open(SPLIT_PATH))

    results = []

    for fold_id in range(5):

        print(f"\n================ FOLD {fold_id} ================")

        train_idx = np.array(folds[fold_id]["train_idx"])
        test_idx  = np.array(folds[fold_id]["test_idx"])

        train_ds = Subset(dataset, train_idx)
        test_ds  = Subset(dataset, test_idx)

        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=16)

        model = SpectralFusionModel().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=3e-4)

        # =========================
        # ⭐ early stopping
        # =========================
        best_loss = float("inf")
        patience = 50
        wait = 0
        best_model = None

        for epoch in range(500):

            model.train()
            total_loss = 0

            for batch in train_loader:
                pred = model(
                    batch["x"].to(device),
                    batch["lambda"].to(device),
                    batch["delta"].to(device),
                    batch["type"].to(device),
                    batch["temp"].to(device)
                )

                loss = r2_aware_loss(pred, batch["y"].to(device))

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item()

            # =========================
            # ⭐ validation = test here
            # =========================
            mse, rmse, mae, r2 = evaluate(model, test_loader, device)

            print(
                f"Epoch {epoch:03d} | "
                f"TrainLoss {total_loss:.4f} | "
                f"Test RMSE {rmse:.4f} R2 {r2:.4f}"
            )

            # =========================
            # ⭐ save best model
            # =========================
            if rmse < best_loss:
                best_loss = rmse
                best_model = copy.deepcopy(model.state_dict())
                wait = 0

                torch.save(
                    best_model,
                    os.path.join(model_save_dir, f"best_model_fold{fold_id}.pth")
                )
            else:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered")
                    break

        # =========================
        # ⭐ final evaluation
        # =========================
        model.load_state_dict(best_model)

        mse, rmse, mae, r2 = evaluate(model, test_loader, device)

        print(f"\n[FOLD {fold_id} FINAL]")
        print(f"MSE: {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
        print(f"R2: {r2:.4f}")

        results.append([mse, rmse, mae, r2])

    # =========================
    # ⭐ overall result
    # =========================
    results = np.array(results)

    print("\n================ FINAL 5-FOLD RESULT ================")
    print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
    print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
    print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
    print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

if __name__ == "__main__":
    train()


================ FOLD 0 ================
Epoch 000 | TrainLoss 1171.4248 | Test RMSE 5.1760 R2 -9.2869
Epoch 001 | TrainLoss 700.1374 | Test RMSE 3.8802 R2 -4.7810
Epoch 002 | TrainLoss 403.0080 | Test RMSE 2.6738 R2 -1.7450
Epoch 003 | TrainLoss 215.5544 | Test RMSE 1.8590 R2 -0.3270
Epoch 004 | TrainLoss 138.2658 | Test RMSE 1.6205 R2 -0.0082
Epoch 005 | TrainLoss 95.9488 | Test RMSE 1.4243 R2 0.2210
Epoch 006 | TrainLoss 63.7885 | Test RMSE 1.4024 R2 0.2449
Epoch 007 | TrainLoss 57.6932 | Test RMSE 1.4336 R2 0.2108
Epoch 008 | TrainLoss 56.8311 | Test RMSE 1.4883 R2 0.1495
Epoch 009 | TrainLoss 57.7905 | Test RMSE 1.5273 R2 0.1044
Epoch 010 | TrainLoss 56.0718 | Test RMSE 1.4805 R2 0.1584
Epoch 011 | TrainLoss 55.0624 | Test RMSE 1.4135 R2 0.2328
Epoch 012 | TrainLoss 55.6294 | Test RMSE 1.5996 R2 0.0175
Epoch 013 | TrainLoss 55.3338 | Test RMSE 1.3331 R2 0.3176
Epoch 014 | TrainLoss 53.9163 | Test RMSE 1.3361 R2 0.3145
Epoch 015 | TrainLoss 50.8493 | Test RMSE 1.2598 R2 0.3907
Epo

### TabPFN

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from tabpfn import TabPFNRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
# Suppress warnings
warnings.filterwarnings("ignore")
# =========================
# path
# =========================
DATA_DIR = r"data/input"
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")

csv_path = os.path.join(DATA_DIR, "UAV_Sentinel_S2A.csv")

# Output directory for predictions
save_dir = os.path.join(DATA_DIR, "UAV-Sentinel2S2A/TabPFN")
os.makedirs(save_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# Load cross-validation splits
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# Evaluation metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# data loading
# =========================
raw = pd.read_csv(csv_path, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = min(11, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "DO(mg/L"

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

# =========================
# group
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print("samples:", len(y), "features:", X.shape[1])
print("group samples:", num_samples)

# =========================
# load folds
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# result container
# =========================
results = []

# =========================
# 5-fold training 
# =========================
for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    # ===== 加标准化 =====
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # =========================
    # model
    # =========================
    model = TabPFNRegressor(
        n_estimators=32,
        device=device
    )

    # =========================
    # train
    # =========================
    model.fit(X_train, y_train)

    # =========================
    # predict
    # =========================
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    # =========================
    # evaluate
    # =========================
    train_metrics = evaluate(y_train, pred_train)
    test_metrics = evaluate(y_test, pred_test)

    print(f"[Train] MSE={train_metrics[0]:.4f} RMSE={train_metrics[1]:.4f} MAE={train_metrics[2]:.4f} R2={train_metrics[3]:.4f}")
    print(f"[Test ] MSE={test_metrics[0]:.4f}  RMSE={test_metrics[1]:.4f}  MAE={test_metrics[2]:.4f} R2={test_metrics[3]:.4f}")

    # =========================
    # save fold result
    # =========================
    results.append(test_metrics)

    np.save(os.path.join(save_dir, f"fold_{fold_id}_pred.npy"), pred_test)

# =========================
# final summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
samples: 570 features: 11
group samples: 57

================ FOLD 0 ================
[Train] MSE=0.0001 RMSE=0.0095 MAE=0.0044 R2=1.0000
[Test ] MSE=0.9589  RMSE=0.9792  MAE=0.6552 R2=0.6318

================ FOLD 1 ================
[Train] MSE=0.0000 RMSE=0.0047 MAE=0.0038 R2=1.0000
[Test ] MSE=2.3754  RMSE=1.5412  MAE=0.9759 R2=0.3042

================ FOLD 2 ================
[Train] MSE=0.0001 RMSE=0.0075 MAE=0.0050 R2=1.0000
[Test ] MSE=0.5053  RMSE=0.7108  MAE=0.5504 R2=0.7930

================ FOLD 3 ================
[Train] MSE=0.0000 RMSE=0.0051 MAE=0.0040 R2=1.0000
[Test ] MSE=0.6631  RMSE=0.8143  MAE=0.6142 R2=0.8080

================ FOLD 4 ================
[Train] MSE=0.0000 RMSE=0.0050 MAE=0.0040 R2=1.0000
[Test ] MSE=4.7205  RMSE=2.1727  MAE=1.2886 R2=-0.1638

================ FINAL 5-FOLD RESULT ================
MSE  : 1.8446 ± 1.5828
RMSE : 1.2437 ± 0.5458
MAE  : 0.8169 ± 0.2779
R2   : 0.4747 ± 0.3671


### Transformer

In [3]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# path
# =========================
DATA_DIR = r"data/input"
CSV_PATH = os.path.join(DATA_DIR, "UAV_Sentinel_S2A.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")
save_dir = os.path.join(DATA_DIR, "UAV-Sentinel2S2A/Transformer")
os.makedirs(save_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# load dataset
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = min(11, raw.shape[1])
target_col = "DO(mg/L"

X_df = raw.iloc[:, :max_feat].copy()

if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print("Samples:", len(y), "Features:", X.shape[1])

# =========================
# group
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

print("Group samples:", num_samples)
# =========================
# load split
# =========================
def load_folds(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# Transformer baseline model
# =========================
class TransformerBaseline(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2):
        super().__init__()

        self.embed = nn.Linear(1, d_model)

        self.pos = nn.Parameter(torch.randn(1, 200, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (B, N)
        x = x.unsqueeze(-1)
        x = self.embed(x)

        N = x.size(1)
        x = x + self.pos[:, :N, :].to(x.device)

        x = self.encoder(x)

        x = x.mean(dim=1)

        return self.head(x).squeeze()
    
# =========================
# load folds
# =========================
folds = load_folds(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # tensor
    # =========================
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    # =========================
    # model
    # =========================
    model = TransformerBaseline().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # =========================
    # training loop
    # =========================
    best_loss = float("inf")
    patience = 50
    wait = 0

    for epoch in range(500):

        model.train()

        pred = model(X_train)
        loss = loss_fn(pred, y_train)

        opt.zero_grad()
        loss.backward()
        opt.step()

        # early stopping
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_model = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # =========================
    # evaluation
    # =========================
    model.load_state_dict(best_model)

    model.eval()
    with torch.no_grad():
        pred = model(X_test).cpu().numpy()

    mse, rmse, mae, r2 = evaluate(y_test, pred)

    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

    results.append([mse, rmse, mae, r2])

    np.save(os.path.join(save_dir, f"fold_{fold_id}_pred.npy"), pred)
    torch.save(best_model, os.path.join(save_dir, f"fold_{fold_id}.pth"))

# =========================
# final result
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
Samples: 570 Features: 11
Group samples: 57

================ FOLD 0 ================
MSE=0.7160, RMSE=0.8462, MAE=0.5526, R2=0.7251

================ FOLD 1 ================
MSE=1.8862, RMSE=1.3734, MAE=0.9540, R2=0.4475

================ FOLD 2 ================
MSE=1.3921, RMSE=1.1799, MAE=0.8300, R2=0.4299

================ FOLD 3 ================
MSE=1.6438, RMSE=1.2821, MAE=1.0003, R2=0.5241

================ FOLD 4 ================
MSE=3.0590, RMSE=1.7490, MAE=1.1799, R2=0.2458

================ FINAL 5-FOLD RESULT ================
MSE  : 1.7394 ± 0.7668
RMSE : 1.2861 ± 0.2921
MAE  : 0.9034 ± 0.2083
R2   : 0.4745 ± 0.1551


### Residual MLP

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# path
# =========================
DATA_DIR = r"data/input"
CSV_PATH = os.path.join(DATA_DIR, "UAV_Sentinel_S2A.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")
save_dir = os.path.join(DATA_DIR, "UAV-Sentinel2S2A/Residual MLP")
os.makedirs(save_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# load dataset
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")

raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = min(11, raw.shape[1])
target_col = "DO(mg/L"

X_df = raw.iloc[:, :max_feat].copy()

if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values.astype(np.float32)
y = data.iloc[:, -1].values.astype(np.float32)

print("Samples:", len(y), "Features:", X.shape[1])

# =========================
# group
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

print("Group samples:", num_samples)

# =========================
# load folds
# =========================
def load_folds(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# MLP model
# =========================
class ResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)
        self.relu = nn.ReLU()

        # Project feature dimensions to a common space if mismatched
        self.proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        return self.relu(self.fc(x) + self.proj(x))


class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()

        self.block1 = ResidualBlock(in_dim, 256)
        self.block2 = ResidualBlock(256, 128)
        self.block3 = ResidualBlock(128, 64)
        self.block4 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return x.squeeze()

# =========================
# folds
# =========================
folds = load_folds(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # tensor
    # =========================
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    # =========================
    # model
    # =========================
    model = MLP(X.shape[1]).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # =========================
    # training loop
    # =========================
    best_loss = float("inf")
    patience = 50
    wait = 0

    for epoch in range(500):

        model.train()

        pred = model(X_train)
        loss = loss_fn(pred, y_train)

        opt.zero_grad()
        loss.backward()
        opt.step()

        # early stopping
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_model = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # =========================
    # evaluation
    # =========================
    model.load_state_dict(best_model)

    model.eval()
    with torch.no_grad():
        pred = model(X_test).cpu().numpy()

    mse, rmse, mae, r2 = evaluate(y_test, pred)

    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

    results.append([mse, rmse, mae, r2])

    np.save(os.path.join(save_dir, f"fold_{fold_id}_pred.npy"), pred)
    torch.save(best_model, os.path.join(save_dir, f"fold_{fold_id}.pth"))

# =========================
# final result
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
Samples: 570 Features: 11
Group samples: 57

================ FOLD 0 ================
MSE=0.7037, RMSE=0.8389, MAE=0.6524, R2=0.7298

================ FOLD 1 ================
MSE=4.7671, RMSE=2.1834, MAE=1.5029, R2=-0.3964

================ FOLD 2 ================
MSE=1.8681, RMSE=1.3668, MAE=0.7766, R2=0.2349

================ FOLD 3 ================
MSE=6.3100, RMSE=2.5120, MAE=1.5945, R2=-0.8267

================ FOLD 4 ================
MSE=3.5522, RMSE=1.8847, MAE=1.3003, R2=0.1242

================ FINAL 5-FOLD RESULT ================
MSE  : 3.4402 ± 1.9984
RMSE : 1.7572 ± 0.5939
MAE  : 1.1653 ± 0.3822
R2   : -0.0268 ± 0.5367


### XGBoost

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import copy

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold
import xgboost as xgb

# =========================
# path config
# =========================
DATA_DIR = r"\data\input"
CSV_PATH = os.path.join(DATA_DIR, "UAV_Sentinel_S2A.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")

SAVE_DIR = os.path.join(DATA_DIR, "UAV-Sentinel2S2A/XGBoost")
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# load data
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip()

# feature selection
max_feat = min(11, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "DO(mg/L"

if target_col not in raw.columns:
    raise ValueError(f"Target column {target_col} not found")

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print(f"Samples: {len(y)}, Features: {X.shape[1]}")

# =========================
# group definition (same as PI-Transformer)
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print(f"Grouped samples: {num_samples}")

# =========================
# load split
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# evaluation
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# load folds
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # model
    # =========================
    model = xgb.XGBRegressor(
        n_estimators=2000,
        max_depth=1,
        learning_rate=0.01,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=1234
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # =========================
    # prediction
    # =========================
    y_pred = model.predict(X_test)

    mse, rmse, mae, r2 = evaluate(y_test, y_pred)

    print(f"Fold {fold_id} | MSE: {mse:.4f} RMSE: {rmse:.4f} MAE: {mae:.4f} R2: {r2:.4f}")

    # =========================
    # save model
    # =========================
    model_path = os.path.join(SAVE_DIR, f"xgb_fold{fold_id}.json")
    model.save_model(model_path)

    print(f"Saved: {model_path}")

    results.append([mse, rmse, mae, r2])

# =========================
# summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE: {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2  : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Samples: 570, Features: 11
Grouped samples: 57

================ FOLD 0 ================
Fold 0 | MSE: 0.9380 RMSE: 0.9685 MAE: 0.6478 R2: 0.6398
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Sentinel2S2A/XGBoost\xgb_fold0.json

================ FOLD 1 ================
Fold 1 | MSE: 1.5734 RMSE: 1.2543 MAE: 0.9511 R2: 0.5391
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Sentinel2S2A/XGBoost\xgb_fold1.json

================ FOLD 2 ================
Fold 2 | MSE: 1.4122 RMSE: 1.1884 MAE: 0.8994 R2: 0.4216
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Sentinel2S2A/XGBoost\xgb_fold2.json

================ FOLD 3 ================
Fold 3 | MSE: 1.7618 RMSE: 1.3273 MAE: 1.0211 R2: 0.4900
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Sentinel2S2A/XGBoost\xgb_fold3.json

================ FOLD 4 ================
Fold 4 | MSE: 2.2320 RMSE: 1.4940 MAE: 1.0392 R2: 0.4497
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Sentinel2S2A/XGBoost\xgb_fold4.json

============

### Random Forest

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import copy
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# path config
# =========================
DATA_DIR = r"\data\input"
CSV_PATH = os.path.join(DATA_DIR, "UAV_Sentinel_S2A.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")

SAVE_DIR = os.path.join(DATA_DIR, "UAV-Sentinel2S2A/RandomForest")
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# load data
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip()

# =========================
# feature / target
# =========================
max_feat = min(11, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "DO(mg/L"

if target_col not in raw.columns:
    raise ValueError(f"Target column {target_col} not found")

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print(f"Samples: {len(y)}, Features: {X.shape[1]}")

# =========================
# group definition (same as PI-Transformer)
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print(f"Grouped samples: {num_samples}")

# =========================
# load split
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# evaluation
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# load splits
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # Random Forest model
    # =========================
    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=True,
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mse, rmse, mae, r2 = evaluate(y_test, y_pred)

    print(f"Fold {fold_id} | MSE: {mse:.4f} RMSE: {rmse:.4f} MAE: {mae:.4f} R2: {r2:.4f}")

    # =========================
    # save model
    # =========================
    model_path = os.path.join(SAVE_DIR, f"rf_fold{fold_id}.pkl")
    joblib.dump(model, model_path)

    print(f"Saved: {model_path}")

    results.append([mse, rmse, mae, r2])

# =========================
# summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE: {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2  : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Samples: 570, Features: 11
Grouped samples: 57

================ FOLD 0 ================
Fold 0 | MSE: 0.7132 RMSE: 0.8445 MAE: 0.5274 R2: 0.7262
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Sentinel2S2A/RandomForest\rf_fold0.pkl

================ FOLD 1 ================
Fold 1 | MSE: 1.4626 RMSE: 1.2094 MAE: 0.8483 R2: 0.5716
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Sentinel2S2A/RandomForest\rf_fold1.pkl

================ FOLD 2 ================
Fold 2 | MSE: 0.8899 RMSE: 0.9433 MAE: 0.6421 R2: 0.6355
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Sentinel2S2A/RandomForest\rf_fold2.pkl

================ FOLD 3 ================
Fold 3 | MSE: 1.2784 RMSE: 1.1307 MAE: 0.8698 R2: 0.6299
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Sentinel2S2A/RandomForest\rf_fold3.pkl

================ FOLD 4 ================
Fold 4 | MSE: 2.7298 RMSE: 1.6522 MAE: 1.1254 R2: 0.3270
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Sentinel2S2A/RandomForest\rf_fold4.pk